# MACE vs CGCNN Surrogate Comparison for Catalyst Discovery

This notebook evaluates both **MACE** (production surrogate) and **CGCNN** (from-scratch baseline) on the same Materials Project catalyst subset, comparing:

- e_above_hull predictions (raw values per material)
- MC-Dropout uncertainty estimates
- Inference speed per candidate
- Stability-classification agreement (both vs 0.1 eV/atom threshold)

**Dataset:** ~130 unique catalyst materials from `data/raw/metadata.json`

**Note:** CGCNN is trained from-scratch on this small subset. The honest limitation (needing far more data to compete with MACE's pretrained foundation) is a defensible and on-thesis result.

---

## Execution

You can either:
1. Run cells sequentially in Jupyter, or
2. Execute the script directly: `python models/gnn_surrogate/MACE_CGCNN_surrogate_comparison.py --train-cgcnn`


## Imports and Configuration

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys_path_root = Path("__file__").resolve().parent.parent
if str(sys_path_root) not in sys.path:
    sys.path.insert(0, str(sys_path_root))

import json
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

from kg.graph_store import load_graph, rehydrate_node
from kg.schema import MaterialNode
from agent.cost_model import SURROGATE_COST

# Try to import CGCNN
try:
    from models.gnn_surrogate.baseline_cgcnn import CGCNN, MaterialsDataset
    from kg.build_graph import build_graph_edges
    CGCNN_AVAILABLE = True
    print("✓ CGCNN module available")
except ImportError as e:
    CGCNN_AVAILABLE = False
    print(f"✗ CGCNN module not found: {e}")
    print("  Run 'python models/gnn_surrogate/baseline_cgcnn.py --train' first")

## Load Catalyst Materials

In [ ]:
# Load KG and filter for catalyst materials with structures
KG_PATH = Path("data/processed/kg.json")

if not KG_PATH.exists():
    raise FileNotFoundError(f"KG file not found: {KG_PATH}. Run 'python kg/build_graph.py' first.")

G = load_graph(KG_PATH)

# Get materials with structures
material_ids = [nid for nid, data in G.nodes(data=True) 
                if data.get("type") == "Material" and data.get("structure_id")]

print(f"Found {len(material_ids)} materials with structures in KG")

# Load up to 130 catalyst materials
materials = []
for mid in material_ids[:130]:
    try:
        mat = rehydrate_node(G, mid)
        if mat.structure_id:
            materials.append(mat)
    except Exception as e:
        print(f"[WARN] Failed to rehydrate {mid}: {e}")
        continue

print(f"Loaded {len(materials)} materials for comparison")
print(f"\nSample materials:")
for mat in materials[:5]:
    print(f"  - {mat.mpid}: {mat.formula_pretty}")

## MACE Predictions (Production Surrogate)

MACE uses the fine-tuned `mace-mpa-0-medium` checkpoint with MC-Dropout for uncertainty estimation.

In [ ]:
from agent.predictor import PredictorAgent

MACE_CHECKPOINT = Path("models/gnn_surrogate/mace-mpa-0-medium.model")

# Run MACE predictions with MC-Dropout
mace_predictions = []

for i, mat in enumerate(materials):
    predictor = PredictorAgent(checkpoint_path=MACE_CHECKPOINT)
    result = predictor.predict(mat)
    
    if result.prediction_failed or result.property_value is None:
        mace_predictions.append({
            "material_id": mat.mpid,
            "formula": mat.formula_pretty,
            "e_above_hull": np.nan,
            "uncertainty": np.nan
        })
    else:
        mace_predictions.append({
            "material_id": mat.mpid,
            "formula": mat.formula_pretty,
            "e_above_hull": float(result.property_value),
            "uncertainty": float(result.uncertainty)
        })
    
    if (i + 1) % 20 == 0:
        print(f"Processed {i+1}/{len(materials)} materials")

print(f"\nMACE predictions complete: {sum(1 for p in mace_predictions if not np.isnan(p['e_above_hull']))} valid predictions")

## CGCNN Training & Predictions (Baseline Surrogate)

CGCNN is trained from-scratch on the ~130-material catalyst subset. This is honest about the data limitation — a real GNN needs far more data to compete with MACE's pretrained foundation.

In [ ]:
if not CGCNN_AVAILABLE:
    raise ImportError("CGCNN module not available. Train it first with: python models/gnn_surrogate/baseline_cgcnn.py --train")

# Load ground truth e_above_hull from KG
data_list = []

for mat in materials:
    # Get property node for this material
    prop_nid = None
    for nid, data in G.nodes(data=True):
        if (data.get("type") == "Property" and 
            data.get("mpid") == mat.mpid and
            data.get("name") == "energy_above_hull"):
            prop_nid = nid
            break
    
    if prop_nid is None:
        continue
        
    props = G.nodes[prop_nid]
    e_above_hull = float(props.get("value", np.nan))
    
    # Get structure and atomic info
    struct_nid = mat.structure_id
    for nid, data in G.nodes(data=True):
        if nid == struct_nid and data.get("type") == "Structure":
            cif_path = Path(data.get("cif_path"))
            if cif_path.exists():
                from pymatgen.core import Structure as PMGStructure
                pmg_struct = PMGStructure.from_file(str(cif_path))
                
                atomic_numbers = np.array([pmg_struct.get_atomic_number(s) 
                                           for s in pmg_struct.species], dtype=np.int64)
                cell = np.array(pmg_struct.lattice.frac_coords, dtype=np.float64)
                positions = np.array(pmg_struct.frac_coords, dtype=np.float64)
                
                data_list.append({
                    "atomic_numbers": atomic_numbers,
                    "cell": cell,
                    "positions": positions,
                    "e_above_hull": e_above_hull,
                    "dataset_type": "uniform"
                })
            break

print(f"Prepared {len(data_list)} training examples")

# Split into train/val (80/20)
np.random.seed(42)
indices = np.arange(len(data_list))
np.random.shuffle(indices)
split = int(0.8 * len(indices))
train_idx, val_idx = indices[:split], indices[split:]

print(f"Train set: {len(train_idx)}, Validation set: {len(val_idx)}")

In [ ]:
# Create datasets and loaders
train_dataset = MaterialsDataset([data_list[i] for i in train_idx])
val_dataset = MaterialsDataset([data_list[i] for i in val_idx])

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=128, shuffle=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=128, shuffle=False
)

# Initialize CGCNN model (medium architecture)
model = CGCNN(hidden_channels=256, num_conv_layers=4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training CGCNN...")
train_losses = []
val_losses = []

# Training loop
for epoch in range(100):
    model.train()
    total_loss = 0.0
    
    for batch in train_loader:
        optimizer.zero_grad()
        batch_indices = torch.zeros(len(batch.x), dtype=torch.long)
        pred = model(batch, batch=batch_indices, training=True)
        target = batch.y
        loss = ((pred - target) ** 2).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    train_losses.append(total_loss / len(train_loader))
    
    # Quick validation
    model.eval()
    with torch.no_grad():
        val_preds = []
        val_targets = []
        for batch in val_loader:
            batch_indices = torch.zeros(len(batch.x), dtype=torch.long)
            pred = model(batch, batch=batch_indices, training=False)
            val_preds.extend(pred.numpy())
            val_targets.extend(batch.y.numpy())
        
        val_preds = np.array(val_preds).flatten()
        val_targets = np.array(val_targets).flatten()
        val_loss = ((val_preds - val_targets) ** 2).mean()
        val_losses.append(val_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/100: train_loss={train_losses[-1]:.4f}, val_loss={val_losses[-1]:.4f}")

# Save model
output_path = Path("models/gnn_surrogate/cgcnn_catalyst.pt")
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'hidden_channels': 256,
        'num_conv_layers': 4,
    },
    'train_losses': train_losses,
    'val_losses': val_losses,
}, output_path)

print(f"\nSaved CGCNN model to {output_path}")

In [ ]:
# Compute final validation metrics
model.eval()
with torch.no_grad():
    val_preds = []
    val_targets = []
    for batch in val_loader:
        batch_indices = torch.zeros(len(batch.x), dtype=torch.long)
        pred = model(batch, batch=batch_indices, training=False)
        val_preds.extend(pred.numpy())
        val_targets.extend(batch.y.numpy())

val_preds = np.array(val_preds).flatten()
val_targets = np.array(val_targets).flatten()

mse = float(np.mean((val_preds - val_targets) ** 2))
mae = float(np.mean(np.abs(val_preds - val_targets)))
rmse = float(np.sqrt(mse))
ss_res = np.sum((val_targets - val_preds) ** 2)
ss_tot = np.sum((val_targets - np.mean(val_targets)) ** 2)
r2 = float(1 - (ss_res / ss_tot)) if ss_tot > 0 else 0.0

print("\nCGCNN Validation Metrics:")
print(f"  MSE: {mse:.4f}")
print(f"  MAE: {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  R²: {r2:.4f}")

## CGCNN Predictions on Test Set

In [ ]:
# Run CGCNN predictions
cgcnn_predictions = []

for i, mat in enumerate(materials):
    struct_nid = mat.structure_id
    for nid, data in G.nodes(data=True):
        if nid == struct_nid and data.get("type") == "Structure":
            cif_path = Path(data.get("cif_path"))
            if not cif_path.exists():
                cgcnn_predictions.append({
                    "material_id": mat.mpid,
                    "formula": mat.formula_pretty,
                    "e_above_hull": np.nan
                })
                break
            
            from pymatgen.core import Structure as PMGStructure
            pmg_struct = PMGStructure.from_file(str(cif_path))
            
            atomic_numbers = np.array([pmg_struct.get_atomic_number(s) 
                                       for s in pmg_struct.species], dtype=np.int64)
            cell = np.array(pmg_struct.lattice.frac_coords, dtype=np.float64)
            positions = np.array(pmg_struct.frac_coords, dtype=np.float64)
            
            # Build graph edges
            edge_index, edge_attr = build_graph_edges(
                torch.tensor(positions, dtype=torch.float),
                torch.tensor(cell, dtype=torch.float)
            )
            
            # Create PyG Data object
            from torch_geometric.data import Data as PyGData
            data = PyGData(
                x=torch.tensor(atomic_numbers, dtype=torch.long),
                edge_index=edge_index,
                edge_attr=edge_attr,
                pos=positions,
                cell=cell,
                y=torch.tensor([float(np.nan)]),
            )
            
            # Predict
            model.eval()
            with torch.no_grad():
                pred = model(data, training=False).item()
            
            cgcnn_predictions.append({
                "material_id": mat.mpid,
                "formula": mat.formula_pretty,
                "e_above_hull": float(pred)
            })
            break
    
    if (i + 1) % 20 == 0:
        print(f"Processed {i+1}/{len(materials)} materials")

print(f"\nCGCNN predictions complete: {sum(1 for p in cgcnn_predictions if not np.isnan(p['e_above_hull']))} valid predictions")

## Comparison Analysis

Now we compare the two models on:
- Raw e_above_hull values
- Stability classification agreement (e_above_hull < 0.1 eV/atom)
- Value correlation between predictions

In [ ]:
# Merge predictions with ground truth
STABILITY_THRESHOLD = 0.1  # eV/atom

comparison_data = []

for i, mat in enumerate(materials):
    # Find ground truth
    gt_value = None
    for nid, data in G.nodes(data=True):
        if (data.get("type") == "Property" and 
            data.get("mpid") == mat.mpid and
            data.get("name") == "energy_above_hull"):
            gt_value = float(data.get("value", np.nan))
            break
    
    # Find MACE prediction
    mace_pred = next((p for p in mace_predictions if p["material_id"] == mat.mpid), None)
    
    # Find CGCNN prediction
    cgcnn_pred = next((p for p in cgcnn_predictions if p["material_id"] == mat.mpid), None)
    
    # Stability classification
    mace_stable = bool(mace_pred and not np.isnan(mace_pred['e_above_hull']) and mace_pred['e_above_hull'] < STABILITY_THRESHOLD)
    cgcnn_stable = bool(cgcnn_pred and not np.isnan(cgcnn_pred['e_above_hull']) and cgcnn_pred['e_above_hull'] < STABILITY_THRESHOLD)
    
    comparison_data.append({
        "material_id": mat.mpid,
        "formula": mat.formula_pretty,
        "ground_truth_e_above_hull": gt_value,
        "mace_stable": mace_stable,
        "cgcnn_stable": cgcnn_stable,
        "mace_e_above_hull": mace_pred['e_above_hull'] if mace_pred else np.nan,
        "cgcnn_e_above_hull": cgcnn_pred['e_above_hull'] if cgcnn_pred else np.nan,
    })

# Compute agreement statistics
mace_stable_flags = [d["mace_stable"] for d in comparison_data]
cgcnn_stable_flags = [d["cgcnn_stable"] for d in comparison_data]

both_stable = sum(1 for ms, cs in zip(mace_stable_flags, cgcnn_stable_flags) if ms and cs)
both_unstable = sum(1 for ms, cs in zip(mace_stable_flags, cgcnn_stable_flags) if not ms and not cs)

agreement_rate = (both_stable + both_unstable) / len(comparison_data) if comparison_data else 0.0

# Value correlation
mace_values = [d["mace_e_above_hull"] for d in comparison_data if not np.isnan(d["mace_e_above_hull"])]
cgcnn_values = [d["cgcnn_e_above_hull"] for d in comparison_data if not np.isnan(d["cgcnn_e_above_hull"])]

common_values = list(set(mace_values) & set(cgcnn_values))
corr = np.nan
if len(common_values) > 1:
    corr, _ = np.polyfit(common_values, common_values, 1)

# Print summary
print("\n" + "="*60)
print("MACE vs CGCNN Comparison Summary")
print("="*60)
print(f"Total materials evaluated: {len(comparison_data)}")
print(f"Stability agreement rate: {agreement_rate*100:.1f}%")
print(f"  Both stable: {both_stable}")
print(f"  Both unstable: {both_unstable}")
print(f"Value correlation (MACE vs CGCNN): {corr:.3f if not np.isnan(corr) else 'N/A'}")

## Visualization

Plots showing:
1. MACE predictions vs ground truth
2. CGCNN predictions vs ground truth
3. Stability classification agreement matrix
4. Scatter plot of MACE vs CGCNN values

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Extract data
mace_eah = [d["mace_e_above_hull"] for d in comparison_data if not np.isnan(d["mace_e_above_hull"])]
cgcnn_eah = [d["cgcnn_e_above_hull"] for d in comparison_data if not np.isnan(d["cgcnn_e_above_hull"])]
gt_eah = [d["ground_truth_e_above_hull"] for d in comparison_data if d["ground_truth_e_above_hull"] is not None]

# Plot 1: MACE vs Ground Truth
ax1 = axes[0, 0]
if gt_eah and mace_eah:
    ax1.scatter(gt_eah, mace_eah, alpha=0.6, s=20, label='MACE', color='steelblue')
    min_val = min(np.min(gt_eah), np.min(mace_eah))
    max_val = max(np.max(gt_eah), np.max(mace_eah))
    ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    ax1.set_xlabel('Ground Truth e_above_hull (eV/atom)')
    ax1.set_ylabel('MACE Predicted e_above_hull (eV/atom)')
    ax1.legend()
    ax1.set_title('MACE: Predicted vs Ground Truth')

# Plot 2: CGCNN vs Ground Truth
ax2 = axes[0, 1]
if gt_eah and cgcnn_eah:
    ax2.scatter(gt_eah, cgcnn_eah, alpha=0.6, s=20, label='CGCNN', color='coral')
    min_val = min(np.min(gt_eah), np.min(cgcnn_eah))
    max_val = max(np.max(gt_eah), np.max(cgcnn_eah))
    ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    ax2.set_xlabel('Ground Truth e_above_hull (eV/atom)')
    ax2.set_ylabel('CGCNN Predicted e_above_hull (eV/atom)')
    ax2.legend()
    ax2.set_title('CGCNN: Predicted vs Ground Truth')

# Plot 3: Stability Agreement Matrix
ax3 = axes[1, 0]
agreement_data = np.array([[both_unstable, 0], [0, both_stable]])
im = ax3.imshow(agreement_data, cmap='Blues', aspect='auto')
ax3.set_xlabel('CGCNN Stable Classification')
ax3.set_ylabel('MACE Stable Classification')
ax3.set_title(f'Stability Agreement Matrix (n={len(comparison_data)})')
ax3.grid(True, which='both', linestyle='--', linewidth=0.5)

# Add counts to cells
for i in range(2):
    for j in range(2):
        ax3.text(j, i, str(agreement_data[i, j]), ha='center', va='center',
                color='white' if agreement_data[i, j] > 10 else 'black')

# Plot 4: MACE vs CGCNN Values
ax4 = axes[1, 1]
if mace_eah and cgcnn_eah:
    ax4.scatter(mace_eah, cgcnn_eah, alpha=0.6, s=20, color='purple')
    min_val = min(np.min(mace_eah), np.min(cgcnn_eah))
    max_val = max(np.max(mace_eah), np.max(cgcnn_eah))
    ax4.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    ax4.set_xlabel('MACE e_above_hull (eV/atom)')
    ax4.set_ylabel('CGCNN e_above_hull (eV/atom)')
    ax4.set_title(f'MACE vs CGCNN Predictions (Correlation: {corr:.3f if not np.isnan(corr) else "N/A"})')

plt.tight_layout()
plt.savefig(Path("notebooks/plots/mace_vs_cgcnn_comparison.png"), dpi=150, bbox_inches='tight')
print("\nSaved plots to notebooks/plots/mace_vs_cgcnn_comparison.png")

## Key Findings

### What to expect with the small dataset (~130 materials):

1. **CGCNN will likely show higher error** than MACE on validation metrics (MSE, MAE, R²)
   - MACE is pretrained on millions of Materials Project structures
   - CGCNN needs far more data to achieve similar performance

2. **Stability classification agreement may be moderate**
   - Both models should agree on clearly stable/unstable materials
   - Disagreement is likely for borderline cases (0.05-0.15 eV/atom)

3. **The honest takeaway**
   - This comparison demonstrates the data-efficiency gap between pretrained MLIPs and from-scratch GNNs
   - It validates that MACE is worth using in production despite being heavier
   - CGCNN serves as a useful baseline for future work with larger datasets

In [ ]:
# Save comparison results to JSON
output_path = Path("models/gnn_surrogate/cgcnn_vs_mace_comparison.json")

comparison_results = {
    "summary": {
        "n_materials": len(comparison_data),
        "stability_agreement_rate": float(agreement_rate),
        "value_correlation": float(corr) if not np.isnan(corr) else None,
    },
    "mace_predictions": mace_predictions,
    "cgcnn_predictions": cgcnn_predictions,
    "comparison_details": comparison_data,
}

with open(output_path, 'w') as f:
    json.dump(comparison_results, f, indent=2, default=str)

print(f"\nSaved comparison results to {output_path}")